<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Creating FABnet IPv4 Network: Manual Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** This notebook walks you through creating a FABnet IPv4 (Layer 3) network connecting two nodes on **different** FABRIC sites, then **manually** assigning IP addresses and routes after the slice becomes active. Manual configuration gives you full control over addressing and is useful when you need specific IP assignments or custom routing.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Create FABnet IPv4 L3 networks **without** pre-configuring interfaces
2. Retrieve the FABRIC-assigned **subnet** and **gateway** from a provisioned network
3. Manually assign IP addresses to interfaces using `iface.ip_addr_add()`
4. Add static routes using `node.ip_route_add()` so nodes on different sites can communicate
5. Verify configuration with `ip addr show` and `ip route list`

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Complete the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be comfortable creating basic slices (see [Hello, FABRIC](../../hello_fabric/hello_fabric.ipynb))

**Tip -- Auto vs. Manual vs. Full Auto:** FABRIC offers three configuration approaches for FABnet:
- **Auto** ([auto notebook](./create_l3network_fabnet_ipv4_auto.ipynb)): You create the networks and interfaces, but FABlib assigns IPs automatically
- **Manual** (this notebook): You create everything and assign IPs yourself after the slice is active
- **Full Auto** ([full auto notebook](./create_l3network_fabnet_ipv4_full_auto.ipynb)): You call `node.add_fabnet()` and FABlib handles everything

</div>

## Background: Manual Configuration of FABnet

With manual configuration, you submit the slice request **without** setting interface modes or adding routes. FABRIC still assigns a subnet and gateway to each L3 network, but it is your responsibility to:

1. **Query** the assigned subnet and available IPs from the network object
2. **Assign** an IP to each interface using `ip_addr_add()`
3. **Add routes** so traffic destined for the remote subnet goes through the local gateway


### When to Use Manual Configuration
- You need **specific IP addresses** for reproducibility or documentation
- You want to understand the underlying networking primitives
- You need **custom routing** beyond simple FABnet connectivity

### NIC Component Models

| Model | Speed | Type | Ports |
|-------|-------|------|-------|
| `NIC_Basic` | 100 Gbps | Mellanox ConnectX-6 SR-IOV VF | 1 |
| `NIC_ConnectX_5` | 25 Gbps | Dedicated Mellanox ConnectX-5 | 2 |
| `NIC_ConnectX_6` | 100 Gbps | Dedicated Mellanox ConnectX-6 | 2 |

## What We're Building

In this notebook we will create two nodes on different sites connected via FABNetv4.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We select two different random sites for cross-site connectivity.

In [ ]:
# Name for the slice
slice_name = 'MySlice'

# Pick two different random FABRIC sites
[site1,site2] = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node and network names
node1_name = 'Node1'
node2_name = 'Node2'

network1_name='net1'
network2_name='net2'

# NIC names (used to identify the component on each node)
node1_nic_name = 'nic1'
node2_nic_name = 'nic2'

## Step 3: Create and Submit the Slice

With manual configuration, we **do not** call `set_mode('auto')` or `add_route()` before submitting. We simply create the nodes, attach NICs, create the L3 networks, and submit.

<div class="fab-danger">

**Important:** All interfaces on a single `l3network` must be on the **same site**. Create one network per site and connect them through FABnet routing.

</div>

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Node1 on site1 with a NIC ---
node1 = slice.add_node(name=node1_name, site=site1)
# Add a NIC_Basic component and get its first (only) interface
iface1 = node1.add_component(model='NIC_Basic', name=node1_nic_name).get_interfaces()[0]

# --- Node2 on site2 with a NIC ---
node2 = slice.add_node(name=node2_name, site=site2)
iface2  = node2.add_component(model='NIC_Basic', name=node2_nic_name).get_interfaces()[0]

# --- Create FABnet IPv4 networks, one per site ---
# Pass the interface list to connect each node to its site's network
net1 = slice.add_l3network(name=network1_name, interfaces=[iface1], type='IPv4')
net2 = slice.add_l3network(name=network2_name, interfaces=[iface2], type='IPv4')

# Submit -- no IP or route configuration yet (that comes after the slice is active)
slice.submit();

## Step 4: Manually Configure IP Addresses

Now that the slice is active, FABRIC has assigned a **subnet** and **gateway** to each network. We need to:
1. Query the assigned subnet and list of available IPs
2. Pick an IP for each node
3. Assign it to the interface
4. Add a route to reach the remote network

### Step 4a: Get the Assigned Subnets

Each FABnet network is assigned its own subnet. Use `get_available_ips()` to see which addresses you can assign.

In [ ]:
# Get the network objects and their available IP addresses
network1 = slice.get_network(name=network1_name)
network1_available_ips = network1.get_available_ips()
network1.show()

network2 = slice.get_network(name=network2_name)
network2_available_ips =  network2.get_available_ips()
network2.show();

### Step 4b: Configure Node1

Get the interface connected to `net1`, assign it an IP from the available pool, and add a route to reach `net2`'s subnet through `net1`'s gateway.

In [ ]:
# Get Node1 and its interface on network1
node1 = slice.get_node(name=node1_name)        
node1_iface = node1.get_interface(network_name=network1_name)  

# Pop the first available IP from network1's pool
node1_addr = network1_available_ips.pop(0)

# Assign the IP address to the interface (with the network's subnet mask)
node1_iface.ip_addr_add(addr=node1_addr, subnet=network1.get_subnet())

# Add a route: to reach network2's subnet, go through network1's gateway
node1.ip_route_add(subnet=network2.get_subnet(), gateway=network1.get_gateway())

# Verify: show the interface configuration and routing table
stdout, stderr = node1.execute(f'ip addr show {node1_iface.get_device_name()}')    
stdout, stderr = node1.execute(f'ip route list')

### Step 4c: Configure Node2

Repeat the same steps for Node2: assign an IP from `net2`'s pool and add a route to `net1`'s subnet.

In [ ]:
# Get Node2 and its interface on network2
node2 = slice.get_node(name=node2_name)        
node2_iface = node2.get_interface(network_name=network2_name) 

# Pop the first available IP from network2's pool
node2_addr = network2_available_ips.pop(0)

# Assign the IP address to the interface
node2_iface.ip_addr_add(addr=node2_addr, subnet=network2.get_subnet())

# Add a route: to reach network1's subnet, go through network2's gateway
node2.ip_route_add(subnet=network1.get_subnet(), gateway=network2.get_gateway())

# Verify: show the interface configuration and routing table
stdout, stderr = node2.execute(f'ip addr show {node2_iface.get_device_name()}')
stdout, stderr = node2.execute(f'ip route list')

## Step 5: Verify Connectivity

Now let us confirm that the manual configuration works by pinging Node2 from Node1.

In [ ]:
# Re-fetch the slice (useful if running this cell independently)
slice = fablib.get_slice(slice_name)

node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)           

# Get Node2's IP from its interface (the one we just assigned)
node2_addr = node2.get_interface(network_name=network2_name).get_ip_addr()

# Ping Node2 from Node1 across the FABnet backbone
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Step 6: Run the Experiment

We will find the ping round trip time for this pair of sites. Your experiment should be more interesting!

In [ ]:
# Run a ping test from Node1 to Node2
node1 = slice.get_node(name=node1_name)        

stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Step 7: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running unnecessarily prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice = fablib.get_slice(name=slice_name)
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `get_available_ips()` returns empty list | All IPs already assigned | Check if another interface is using the subnet; try a fresh slice |
| `ip_addr_add()` fails | Interface not found or wrong network name | Verify `network_name` matches the name used in `add_l3network()` |
| `ping` fails one direction only | Missing route on one node | Ensure both nodes have `ip_route_add()` pointing to the other subnet |
| `ping` fails both directions | IPs not assigned or interfaces down | Run `ip addr show` and `ip route list` to verify configuration |
| Slice stuck in `Configuring` | Site may be busy or down | Try different sites by re-running `get_random_sites()` |
| `No resources available` | Site lacks NIC capacity | Choose a different site or use `NIC_ConnectX_6` |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.get_random_sites(count)` | Get a list of distinct random site names | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `fablib.get_slice(name)` | Retrieve an existing slice by name | [get_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_slice) |
| `slice.add_l3network(name, interfaces, type)` | Add a Layer 3 network to the slice | [add_l3network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_l3network) |
| `slice.add_node(name, site)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.get_network(name)` | Get a network object by name | [get_network](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_network) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |
| `node.add_component(model, name)` | Add a NIC or other component | [add_component](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_component) |
| `node.get_interface(network_name)` | Get the interface connected to a network | [get_interface](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.get_interface) |
| `node.ip_route_add(subnet, gateway)` | Add a static route on the node | [ip_route_add](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.ip_route_add) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `iface.ip_addr_add(addr, subnet)` | Assign an IP address to an interface | [ip_addr_add](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.ip_addr_add) |
| `iface.get_ip_addr()` | Get the IP address assigned to an interface | [get_ip_addr](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_ip_addr) |
| `network.get_available_ips()` | Get list of available IPs in the subnet | [get_available_ips](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_available_ips) |
| `network.get_subnet()` | Get the subnet assigned to the network | [get_subnet](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_subnet) |
| `network.get_gateway()` | Get the gateway IP for the network | [get_gateway](https://fabric-fablib.readthedocs.io/en/latest/network_service.html#fabrictestbed_extensions.fablib.network_service.NetworkService.get_gateway) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **FABnet IPv4 Auto** | [create_l3network_fabnet_ipv4_auto](./create_l3network_fabnet_ipv4_auto.ipynb) | Let FABlib assign IPs automatically |
| **FABnet IPv4 Full Auto** | [create_l3network_fabnet_ipv4_full_auto](./create_l3network_fabnet_ipv4_full_auto.ipynb) | Use `add_fabnet()` for the simplest setup |
| **L2 Local Network** | [create_l2network_basic](../create_l2network_basic/create_l2network_basic_manual.ipynb) | Create a Layer 2 Ethernet with manual IPs |
| **L2 Wide-Area Network** | [create_l2network_wide_area](../create_l2network_wide_area/create_l2network_wide_area_manual.ipynb) | Create a WAN Layer 2 circuit with manual IPs |
| **FABnet IPv6** | [create_l3network_fabnet_ipv6](../create_l3network_fabnet_ipv6/create_l3network_fabnet_ipv6_manual.ipynb) | Use FABnet with IPv6 addressing |